# Bike Sharing Demand Prediction
## Step 1: Data Loading & Preprocessing
**Name:** Nitin | **Roll No:** 2323008

In this notebook, we load the raw bike-sharing dataset, explore it, clean it, and prepare it for modelling.

In [8]:
# Importing required libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [20]:
# Load the raw daily dataset
raw_data = pd.read_csv("Datasets/day.csv")
print('Dataset Shape:', raw_data.shape)
raw_data.head()

Dataset Shape: (731, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [10]:
# Check all available columns
print('Columns in dataset:')
for col in raw_data.columns:
    print(' -', col)

Columns in dataset:
 - instant
 - dteday
 - season
 - yr
 - mnth
 - holiday
 - weekday
 - workingday
 - weathersit
 - temp
 - atemp
 - hum
 - windspeed
 - casual
 - registered
 - cnt


In [11]:
# Basic statistical summary
raw_data.describe().T

,count,mean,std,min,25%,50%,75%,max
instant,731.0,366.000000,211.165812,1.000000,183.500000,366.000000,548.500000,731.000000
season,731.0,2.496580,1.110807,1.000000,2.000000,3.000000,3.000000,4.000000
yr,731.0,0.500684,0.500342,0.000000,0.000000,1.000000,1.000000,1.000000
mnth,731.0,6.519836,3.451913,1.000000,4.000000,7.000000,10.000000,12.000000
holiday,731.0,0.028728,0.167155,0.000000,0.000000,0.000000,0.000000,1.000000
weekday,731.0,2.997264,2.004787,0.000000,1.000000,3.000000,5.000000,6.000000
workingday,731.0,0.683995,0.465233,0.000000,0.000000,1.000000,1.000000,1.000000
weathersit,731.0,1.395349,0.544894,1.000000,1.000000,1.000000,2.000000,3.000000
temp,731.0,0.495385,0.183051,0.059130,0.337083,0.498333,0.655417,0.861667
atemp,731.0,0.474354,0.162961,0.079070,0.337842,0.486733,0.608602,0.840896


In [12]:
# Check for missing values
missing = raw_data.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

Missing values per column:
No missing values found!


In [13]:
# Check for duplicate rows
dup_count = raw_data.duplicated().sum()
print(f'Total duplicate rows: {dup_count}')

Total duplicate rows: 0


In [14]:
# Drop columns not needed for prediction
# 'instant' = row index, 'dteday' = date string, 
# 'casual' & 'registered' are subsets of 'cnt' (target leakage)
# 'atemp' = feels-like temp, highly correlated with temp
cols_to_remove = ['instant', 'dteday', 'casual', 'registered', 'atemp']
bike_data = raw_data.drop(columns=cols_to_remove)
print('Remaining columns:', list(bike_data.columns))

Remaining columns: ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed', 'cnt']


In [15]:
# Rename columns for readability
bike_data.rename(columns={
    'cnt': 'total_rentals',
    'mnth': 'month',
    'yr': 'year',
    'hum': 'humidity',
    'weathersit': 'weather'
}, inplace=True)
bike_data.head()

,season,year,month,holiday,weekday,workingday,weather,temp,humidity,windspeed,total_rentals
0,1,0,1,0,6,0,2,0.344167,0.805833,0.160446,985
1,1,0,1,0,0,0,2,0.363478,0.696087,0.248539,801
2,1,0,1,0,1,1,1,0.196364,0.437273,0.248309,1349
3,1,0,1,0,2,1,1,0.200000,0.590435,0.160296,1562
4,1,0,1,0,3,1,1,0.226957,0.436957,0.186900,1600


In [16]:
# Mark categorical columns appropriately
cat_cols = ['season', 'year', 'month', 'holiday', 'workingday', 'weather']
bike_data[cat_cols] = bike_data[cat_cols].astype('category')
bike_data.dtypes

season           category
year             category
month            category
holiday          category
weekday             int64
workingday       category
weather          category
temp              float64
humidity          float64
windspeed         float64
total_rentals       int64
dtype: object

In [17]:
# Quick summary stats on target variable
print('Max rentals in a single day:', bike_data['total_rentals'].max())
print('Min rentals in a single day:', bike_data['total_rentals'].min())
print('Average daily rentals:', round(bike_data['total_rentals'].mean(), 2))

Max rentals in a single day: 8714
Min rentals in a single day: 22
Average daily rentals: 4504.35


In [18]:
# Quick insight: average rentals by season
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
season_avg = bike_data.groupby('season')['total_rentals'].mean().rename(index=season_map)
print('Average Rentals by Season:')
print(season_avg)

Average Rentals by Season:
season
Spring    2604.132597
Summer    4992.331522
Fall      5644.303191
Winter    4728.162921
Name: total_rentals, dtype: float64


In [19]:
# Export cleaned dataset for next stage
bike_data.to_csv('final_dataset.csv', index=False)
print('Cleaned dataset saved as final_dataset.csv')

Cleaned dataset saved as final_dataset.csv
